# shadowclock - local VLM evaluation on Kaggle (clean structure, no API key)

## Required structure (exactly two datasets attached, nothing else under /kaggle/input)
```
shadowclock-code/            <- FLAT folder, these 7 files only (no sub-folders, no .venv, no data, no duplicates)
    run_vlm_local.py  evaluate_hypotheses.py  score.py  solar.py  make_pairs.py  make_probes.py  baseline.py
shadowclock-data/            <- the rendered dataset
    images/                  (720 PNGs)
    metadata.jsonl
```
Do NOT upload pairs/probes files: this notebook regenerates them (deterministically, with the current prompt version)
from `metadata.jsonl` every session, so a stale pairs file can never be used by accident.
Settings: Accelerator = GPU (T4 x2 or P100), Internet = On. In the Input panel REMOVE any old datasets
(e.g. the old `py-files`), otherwise the setup cell stops and lists the duplicates.
Run cells top to bottom. Only the config cell is meant to be edited.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
# ---- environment health check (the runner imports transformers; a broken numpy/scipy/sklearn stack kills it) ----
import subprocess, sys
chk = subprocess.run([sys.executable, "-c",
    "import numpy, scipy, scipy.stats, sklearn, PIL; "
    "from transformers import AutoModelForImageTextToText, AutoProcessor; import transformers, torch; "
    "print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| sklearn', sklearn.__version__, "
    "'| transformers', transformers.__version__, '| torch', torch.__version__)"], capture_output=True, text=True)
print(chk.stdout.strip() or chk.stderr[-1800:])
if chk.returncode:
    raise RuntimeError("The Python environment is broken (see the error above). Stop this session and start a new one "
                       "(Run menu -> stop/restart session [VERIFY name]), then run the notebook from the top again. "
                       "Do not pip-install numpy or scipy.")

In [ ]:
# ---- config: the only cell you edit ----
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"   # Hugging Face model id used by the smoke test / pilot / full run
LABEL = "qwen3b"                         # short name used in file names and result tables
EXTRA = ""                               # e.g. "--load-4bit" (7B on one GPU) or "--dtype bf16"
SAMPLE = 100                             # pairs in the pilot (twins kept together)
DATA_FILTER = ""                         # when TWO data datasets are attached (synthetic + real), put a substring of the one to use, e.g. "real"
BASELINE_METHOD = "components"           # "components" (simple boxes), "boundary" or "auto" (use "auto" for the real-geometry scenes)

In [ ]:
# ---- SETUP: strict discovery + fresh pairs/probes + baseline. Run first in EVERY session. ----
import hashlib, json, os, shlex, subprocess, sys, time
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
GEN = WORK / "generated"
REQUIRED = ["run_vlm_local.py", "evaluate_hypotheses.py", "score.py", "solar.py",
            "make_pairs.py", "make_probes.py", "baseline.py"]
RUN_BASELINE = True

def unique(name, must_have=None, contains=None):
    hits = [p for p in sorted(INPUT.rglob(name)) if (must_have is None or (p.parent / must_have).exists())
            and (not contains or contains in str(p))]
    if len(hits) != 1:
        listing = "\n  ".join(str(h) for h in hits) or "(none)"
        raise RuntimeError(
            f"Expected exactly ONE {name}" + (f" next to a folder '{must_have}'" if must_have else "")
            + f" under {INPUT}, found {len(hits)}:\n  {listing}\n"
            "Fix: in the Input panel remove old/duplicate datasets, or delete the duplicate files.")
    return hits[0]

CODE = unique("run_vlm_local.py").parent
for f in REQUIRED:
    assert unique(f).parent == CODE, f"{f} must sit in the same folder as run_vlm_local.py ({CODE})"
DATA = unique("metadata.jsonl", must_have="images", contains=DATA_FILTER).parent

def sh(args, tail=1200):
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    print((r.stdout + r.stderr)[-tail:])
    if r.returncode:
        raise RuntimeError(f"command failed (exit {r.returncode}): {' '.join(str(a) for a in args)}")

GEN.mkdir(parents=True, exist_ok=True)
PAIRS, PROBES, BASELINE = GEN / "pairs.jsonl", GEN / "probes.jsonl", GEN / "preds_baseline.jsonl"
sh([sys.executable, CODE / "make_pairs.py", "--data", DATA, "--out", PAIRS, "--dt", "1,2,4", "--max-per-dt", "2",
    "--add-swapped", "--max-el", "70", "--prompt-version", "v2"])
sh([sys.executable, CODE / "make_probes.py", "--data", DATA, "--out", PROBES, "--n", "200", "--max-el", "70",
    "--seed", "0", "--prompt-version", "v2"])

# ---- hard checks: prompts must be v2 (no example answer) and every referenced image must exist ----
first = json.loads(open(PAIRS).readline())
assert first.get("prompt_version") == "v2", f"pairs prompt_version is {first.get('prompt_version')!r}, expected 'v2'"
assert "for example" not in first["prompts"]["R1"], "an example answer is still inside the pair prompt"
q0 = json.loads(open(PROBES).readline())
assert q0.get("prompt_version") == "v2" and "for example" not in q0["prompt"], "probe prompt is not v2"
imgs = set()
for l in open(PAIRS):
    p = json.loads(l)
    imgs.update([p["image_a"], p["image_b"]])
for l in open(PROBES):
    imgs.add(json.loads(l)["image"])
missing = [i for i in sorted(imgs) if not (DATA / i).exists()]
assert not missing, f"{len(missing)} referenced images are missing from {DATA}, e.g. {missing[:3]}"
N_PAIRS = sum(1 for _ in open(PAIRS)); N_PROBES = sum(1 for _ in open(PROBES))
TAG = hashlib.md5(open(PAIRS, "rb").read() + open(PROBES, "rb").read()).hexdigest()[:8]  # changes if prompts change
if RUN_BASELINE and not BASELINE.exists():
    sh([sys.executable, CODE / "baseline.py", "--data", DATA, "--pairs", PAIRS, "--out", BASELINE, "--method", BASELINE_METHOD], tail=600)

# ---- helpers used by the later cells ----
def stream(cmd):
    """Run a command, print only the runner's [local] progress lines live, dump the tail if it fails."""
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail.append(line)
        if line.startswith("[local]") and "device map" not in line:
            print(line.rstrip(), flush=True)
    p.wait()
    if p.returncode:
        print("".join(tail[-25:]))
    return p.returncode

def run_model(model, label, out, extra="", args=()):
    return stream([sys.executable, CODE / "run_vlm_local.py", "--data", DATA, "--pairs", PAIRS, "--probes", PROBES,
                   "--model", model, "--label", label, "--out", out, *args, *shlex.split(extra)])

def evaluate(files, partial=False):
    cmd = [sys.executable, CODE / "evaluate_hypotheses.py", "--pairs", PAIRS, "--probes", PROBES]
    if BASELINE.exists():
        cmd += ["--preds", BASELINE]
    for f in files:
        cmd += ["--preds", f]
    if partial:
        cmd.append("--allow-partial")
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr[-2000:])

print("\n" + "=" * 70)
print("code folder :", CODE)
print("data folder :", DATA, f"({len(list((DATA / 'images').glob('*.png')))} PNGs)")
print(f"pairs       : {N_PAIRS} (928 for the default 8-scene synthetic set; other datasets differ)   probes: {N_PROBES} (expect 200)")
print("prompt      : v2, no example answer   | file tag (changes if prompts change):", TAG)
print("baseline    :", BASELINE if BASELINE.exists() else "not computed")
try:
    import torch
    print("GPUs        :", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "NONE - enable a GPU")
except Exception as exc:  # noqa: BLE001
    print("torch check skipped:", exc)

## 1. Smoke test: 2 pairs x 3 regimes + 4 probes. Read the raw answers (and any error text) before going further.

In [ ]:
smoke = WORK / f"smoke_{LABEL}_{TAG}.jsonl"
if smoke.exists():
    smoke.unlink()
run_model(MODEL, LABEL, smoke, EXTRA, ["--max-pairs", "2", "--probe-max", "4"])
if not smoke.exists():
    raise RuntimeError("The runner produced no output file - read the traceback printed above (usually an environment or model-loading problem).")
for l in open(smoke):
    r = json.loads(l)
    print(f"{r['regime']:5s} {r['id']:11s} -> {r['response'][:150]!r} {('ERROR: ' + r['error']) if r.get('error') else ''}")

## 2. Pilot: random pairs (twins kept together) + all probes, then evaluate against the frozen thresholds

In [ ]:
pilot = WORK / f"pilot_{LABEL}_{TAG}.jsonl"
run_model(MODEL, LABEL, pilot, EXTRA, ["--sample", str(SAMPLE)])
evaluate([pilot], partial=True)

## 3. Mini-pilot on several models (R1 only: 40 pairs + 100 probes each). A model that fails is reported with its error and skipped.

In [ ]:
MODELS = [  # (Hugging Face id, short label, extra runner flags) - ids NOT verified; check each on huggingface.co
    ("Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b", ""),                       # if answers are garbage/NaN on T4: "--load-4bit"
    ("llava-hf/llava-onevision-qwen2-7b-ov-hf", "llavaov7b", "--resize 384"),  # 512 px images ran out of memory; DISCLOSE the resize
    # ("HuggingFaceTB/SmolVLM-Instruct", "smolvlm", ""),                 # failed the output-format gate (100% invalid), skipped
]
finished = []
for hf_id, label, extra in MODELS:
    out = WORK / f"mini_{label}_{TAG}.jsonl"
    print(f"\n=== {label} ({hf_id}) ===", flush=True)
    run_model(hf_id, label, out, extra, ["--regimes", "R1", "--sample", "40", "--probe-max", "100"])
    if out.exists():
        rows = [json.loads(l) for l in open(out)]
        errs = [r["error"] for r in rows if r.get("error")]
        print(f"  answers: {len(rows)}   failed requests: {len(errs)}" + (f"   first error: {errs[0]}" if errs else ""))
        if len(errs) < len(rows):
            finished.append(out)
if finished:
    print("\nEVALUATION (R1 only: H1 and H3 print NOT SCORED by design)")
    evaluate(finished, partial=True)

## 3b. EXPLORATORY larger time gaps (6 h and 8 h), regime R2 (latitude/date given). NOT R1: with only the hemisphere known, ordering is ambiguous once the sun azimuth changes by more than 180 degrees, so even the analytic rule drops to ~75% (6 h) and ~60% (8 h) in R1. Not part of the frozen hypotheses.

In [ ]:
LADDER_MODEL, LADDER_LABEL, LADDER_EXTRA = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b_ladder", ""
ladder = GEN / "pairs_ladder.jsonl"
sh([sys.executable, CODE / "make_pairs.py", "--data", DATA, "--out", ladder, "--dt", "6,8", "--max-per-dt", "2",
    "--add-swapped", "--max-el", "70", "--prompt-version", "v2"])
ladder_base = GEN / "preds_baseline_ladder.jsonl"   # ids differ from the main pairs file, so it needs its own baseline
sh([sys.executable, CODE / "baseline.py", "--data", DATA, "--pairs", ladder, "--out", ladder_base, "--method", BASELINE_METHOD], tail=300)
ladder_out = WORK / f"ladder_{LADDER_LABEL}_{TAG}.jsonl"
stream([sys.executable, CODE / "run_vlm_local.py", "--data", DATA, "--pairs", ladder, "--model", LADDER_MODEL,
        "--label", LADDER_LABEL, "--out", ladder_out, "--regimes", "R2", "--sample", "60", *shlex.split(LADDER_EXTRA)])
r = subprocess.run([str(c) for c in [sys.executable, CODE / "evaluate_hypotheses.py", "--pairs", ladder, "--preds", ladder_base,
                                      "--preds", ladder_out, "--allow-partial"]], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-1500:] if r.returncode else "")

## 3c. EXPLORATORY decomposition (R3): the true shadow bearings are STATED in the prompt, so perception is removed. Same pairs/ids as the main file. Run for the model(s) you care about.

In [ ]:
R3_MODEL, R3_LABEL, R3_EXTRA = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b_r3", ""
pairs_r3 = GEN / "pairs_r3.jsonl"   # identical ids/truths/R0-R2 prompts to pairs.jsonl, plus an R3 prompt
sh([sys.executable, CODE / "make_pairs.py", "--data", DATA, "--out", pairs_r3, "--dt", "1,2,4", "--max-per-dt", "2",
    "--add-swapped", "--max-el", "70", "--prompt-version", "v2", "--with-azimuth"], tail=300)
r3_out = WORK / f"r3_{R3_LABEL}_{TAG}.jsonl"
stream([sys.executable, CODE / "run_vlm_local.py", "--data", DATA, "--pairs", pairs_r3, "--model", R3_MODEL,
        "--label", R3_LABEL, "--out", r3_out, "--regimes", "R3", "--sample", "100", *shlex.split(R3_EXTRA)])
r = subprocess.run([str(c) for c in [sys.executable, CODE / "evaluate_hypotheses.py", "--pairs", pairs_r3,
                                      "--preds", r3_out, "--allow-partial"]], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-1500:] if r.returncode else "")

## 3d. EXPLORATORY chain-of-thought: the model may reason step by step before the final JSON line (the JSON-only prompts forbid this). Regimes R1 and R3, 60 pairs. Fairness check before claiming the model cannot do the task.

In [ ]:
COT_MODEL, COT_LABEL, COT_EXTRA = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b_cot", ""
pairs_cot = GEN / "pairs_cot.jsonl"   # same ids/truths as pairs.jsonl; prompts ask for brief step-by-step reasoning first
sh([sys.executable, CODE / "make_pairs.py", "--data", DATA, "--out", pairs_cot, "--dt", "1,2,4", "--max-per-dt", "2",
    "--add-swapped", "--max-el", "70", "--prompt-version", "v2cot", "--with-azimuth"], tail=300)
cot_out = WORK / f"cot_{COT_LABEL}_{TAG}.jsonl"
stream([sys.executable, CODE / "run_vlm_local.py", "--data", DATA, "--pairs", pairs_cot, "--model", COT_MODEL,
        "--label", COT_LABEL, "--out", cot_out, "--regimes", "R1,R3", "--sample", "60", "--max-new-tokens", "400",
        *shlex.split(COT_EXTRA)])
r = subprocess.run([str(c) for c in [sys.executable, CODE / "evaluate_hypotheses.py", "--pairs", pairs_cot,
                                      "--preds", cot_out, "--allow-partial"]], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-1500:] if r.returncode else "")
# show two raw answers so you can read the reasoning yourself
for l in list(open(cot_out))[:2]:
    print("-" * 60); print(json.loads(l)["response"][:700])

## 3g. EXPLORATORY R4 with step-by-step reasoning allowed (the fairest test of 'can the model apply an explicit rule?'). 60 pairs, 400 new tokens, about 10 s per request.

In [ ]:
COT4_MODEL, COT4_LABEL, COT4_EXTRA = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b_cot4", ""
pairs_cot = GEN / "pairs_cot.jsonl"   # regenerate so it contains the R4 prompt in the step-by-step wording
sh([sys.executable, CODE / "make_pairs.py", "--data", DATA, "--out", pairs_cot, "--dt", "1,2,4", "--max-per-dt", "2",
    "--add-swapped", "--max-el", "70", "--prompt-version", "v2cot", "--with-azimuth"], tail=300)
assert "R4" in json.loads(open(pairs_cot).readline())["prompts"], "no R4 prompt: re-upload the updated make_pairs.py and restart"
cot4_out = WORK / f"cot4_{COT4_LABEL}_{TAG}.jsonl"
stream([sys.executable, CODE / "run_vlm_local.py", "--data", DATA, "--pairs", pairs_cot, "--model", COT4_MODEL,
        "--label", COT4_LABEL, "--out", cot4_out, "--regimes", "R4", "--sample", "60", "--max-new-tokens", "400",
        *shlex.split(COT4_EXTRA)])
r = subprocess.run([str(c) for c in [sys.executable, CODE / "evaluate_hypotheses.py", "--pairs", pairs_cot,
                                      "--preds", cot4_out, "--allow-partial"]], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-1500:] if r.returncode else "")
for l in list(open(cot4_out))[:2]:
    print("-" * 60); print(json.loads(l)["response"][:900])


## 3e. DIAGNOSTIC: does the model follow a simple wrong rule ('later = larger shadow-bearing number', ignoring hemisphere and the 0/360 wrap)? Needs the outputs of 3c/3d/3f; R1 is the negative control.

In [ ]:
import re
meta = {json.loads(l)["image"].replace("\\", "/"): json.loads(l) for l in open(DATA / "metadata.jsonl")}
def shadow_bearing(img):
    return (meta[img.replace("\\", "/")]["sun_az_deg"] + 180.0) % 360.0
def last_answer(text):
    for m in reversed(re.findall(r"\{[^{}]*\}", text or "")):
        try:
            e = str(json.loads(m).get("earlier", "")).strip().upper()[:1]
        except Exception:
            continue
        if e in ("A", "B"):
            return e
    return None
checks = []
for name, pname in (("cot_out", "pairs_cot"), ("r3_out", "pairs_r3"), ("r4_out", "pairs_r3"), ("cot4_out", "pairs_cot")):
    if name in globals() and globals()[name].exists():
        checks.append((name, globals()[pname], globals()[name]))
for name, pf, of in checks:
    pairs_by_id = {json.loads(l)["id"]: json.loads(l) for l in open(pf)}
    for regime in ("R1", "R3", "R4"):
        n = agree = a_cnt = correct = naive_correct = n_naiveA = a_given_nA = a_given_nB = 0
        for l in open(of):
            r = json.loads(l)
            if r["regime"] != regime or r["id"] not in pairs_by_id:
                continue
            ans = last_answer(r["response"])
            if ans is None:
                continue
            p = pairs_by_id[r["id"]]
            naive = "A" if shadow_bearing(p["image_a"]) < shadow_bearing(p["image_b"]) else "B"   # later = larger bearing
            truth = "A" if p["a_earlier"] else "B"
            n += 1; agree += ans == naive; a_cnt += ans == "A"; correct += ans == truth; naive_correct += naive == truth
            if naive == "A":
                n_naiveA += 1; a_given_nA += ans == "A"
            else:
                a_given_nB += ans == "A"
        if n:
            n_naiveB = n - n_naiveA
            print(f"{name:8s} {regime}: n={n:3d}  agrees with naive rule {100 * agree / n:3.0f}%  answers 'A' {100 * a_cnt / n:3.0f}%  correct {100 * correct / n:3.0f}%")
            print(f"           naive rule's own accuracy on these pairs {100 * naive_correct / n:.0f}%;  P(model says A | naive says A) = "
                  f"{100 * a_given_nA / max(n_naiveA, 1):.0f}%,  P(model says A | naive says B) = {100 * a_given_nB / max(n_naiveB, 1):.0f}%")
print("reading: equal 'says A' rates in the two cases => pure position bias; a large gap => the answers respond to the bearing comparison. "
      "R1 is a control: the bearings are NOT in the R1 prompt, so agreement there should be about 50%.")


## 3f. EXPLORATORY R4: bearings AND the sun-motion rule stated in the prompt. Separates a missing rule (R4 high) from an inability to apply a rule (R4 still low).

In [ ]:
R4_MODEL, R4_LABEL, R4_EXTRA = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b_r4", ""
pairs_r3 = GEN / "pairs_r3.jsonl"   # always regenerate: it must contain the R3 AND R4 prompts (an old file from an earlier session may not)
sh([sys.executable, CODE / "make_pairs.py", "--data", DATA, "--out", pairs_r3, "--dt", "1,2,4", "--max-per-dt", "2",
    "--add-swapped", "--max-el", "70", "--prompt-version", "v2", "--with-azimuth"], tail=300)
assert "R4" in json.loads(open(pairs_r3).readline())["prompts"], "no R4 prompt: re-upload the updated make_pairs.py to shadowclock-code and restart"
r4_out = WORK / f"r4_{R4_LABEL}_{TAG}.jsonl"
stream([sys.executable, CODE / "run_vlm_local.py", "--data", DATA, "--pairs", pairs_r3, "--model", R4_MODEL,
        "--label", R4_LABEL, "--out", r4_out, "--regimes", "R4", "--sample", "100", *shlex.split(R4_EXTRA)])
r = subprocess.run([str(c) for c in [sys.executable, CODE / "evaluate_hypotheses.py", "--pairs", pairs_r3,
                                      "--preds", r4_out, "--allow-partial"]], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-1500:] if r.returncode else "")


## 4a. MAIN NUMBERS for the paper: R0, R1, R2 on a random sample of pairs plus all 200 probes, several models, evaluated against the frozen thresholds (H1-H4). Run the models one at a time if your GPU quota is tight (delete entries you do not need). Expect roughly 40 min for a 3B/7B Qwen model [VERIFY].

In [ ]:
FINAL_MODELS = [  # (Hugging Face id, label, extra runner flags, pairs to sample). R0, R1, R2 for every sampled pair plus ALL 200 probes.
    ("Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b_main", "", 300),
    ("Qwen/Qwen2.5-VL-3B-Instruct", "qwen3b_main", "", 300),
    ("llava-hf/llava-onevision-qwen2-7b-ov-hf", "llavaov7b_main", "--resize 384", 100),  # slow (about 8 s per request); DISCLOSE the resize
]
main_done = []
for hf_id, label, extra, n_pairs in FINAL_MODELS:
    out = WORK / f"main_{label}_{TAG}.jsonl"
    print(f"\n=== {label} ({hf_id}), {n_pairs} pairs x 3 regimes + {N_PROBES} probes ===", flush=True)
    run_model(hf_id, label, out, extra, ["--sample", str(n_pairs)])
    if out.exists():
        rows = [json.loads(l) for l in open(out)]
        errs = [r["error"] for r in rows if r.get("error")]
        print(f"  answers: {len(rows)}   failed requests: {len(errs)}" + (f"   first error: {errs[0]}" if errs else ""))
        main_done.append(out)
if main_done:
    print("\nEVALUATION against the frozen thresholds (each model is scored on its own sampled pairs)")
    evaluate(main_done, partial=True)


## 4b. MORE MODELS: each candidate is first screened on 4 pairs (must load and answer in the required JSON format), then run for R0-R2 on a random sample plus all probes. Models that fail are skipped with the reason. Roughly 35-45 min per 4B-8B model at your measured speeds [VERIFY]; check your weekly GPU quota and delete entries you do not need.

In [ ]:
ZOO = [  # (Hugging Face id, label, extra runner flags, pairs to sample). Listed in priority order. Ids were checked on Hugging Face search results, not loaded.
    ("Qwen/Qwen3-VL-8B-Instruct", "qwen3vl8b", "", 200),
    ("OpenGVLab/InternVL3-8B-hf", "internvl3_8b", "--resize 448", 200),   # tiling can explode the token count; DISCLOSE the resize
    ("google/gemma-4-E4B-it", "gemma4_e4b", "", 200),
    ("Qwen/Qwen3-VL-4B-Instruct", "qwen3vl4b", "", 200),                  # id NOT verified
]
# If answers are garbage/NaN on the T4 for a model, retry it with extra flag "--load-4bit" or "--dtype bf16".

def pair_answers_valid(path):
    ok = tot = 0
    for l in open(path):
        r = json.loads(l)
        if r["regime"] != "R1":
            continue
        tot += 1
        for m in reversed(re.findall(r"\{[^{}]*\}", r.get("response", "") or "")):
            try:
                if str(json.loads(m).get("earlier", "")).strip().upper()[:1] in ("A", "B"):
                    ok += 1
                    break
            except Exception:
                pass
    return ok, tot

zoo_done = []
for hf_id, label, extra, n_pairs in ZOO:
    print(f"\n=== {label} ({hf_id}) ===", flush=True)
    smoke = WORK / f"zoo_smoke_{label}_{TAG}.jsonl"
    if smoke.exists():
        smoke.unlink()
    run_model(hf_id, label, smoke, extra, ["--max-pairs", "4", "--regimes", "R1", "--probe-max", "1"])
    if not smoke.exists():
        print("  SKIPPED: the model did not load or the runner crashed (see the messages above).")
        continue
    ok, tot = pair_answers_valid(smoke)
    errs = [json.loads(l).get("error") for l in open(smoke) if json.loads(l).get("error")]
    print(f"  screening: {ok}/{tot} pair answers parse as the required JSON" + (f"; first error: {errs[0]}" if errs else ""))
    if tot == 0 or ok < tot:
        print("  SKIPPED: failed the output-format screening (report it as 'not scored', as with SmolVLM).")
        continue
    out = WORK / f"main_{label}_{TAG}.jsonl"
    print(f"  main run: {n_pairs} pairs x 3 regimes + all probes", flush=True)
    run_model(hf_id, label + "_main", out, extra, ["--sample", str(n_pairs)])
    if out.exists():
        zoo_done.append(out)
if zoo_done:
    print("\nEVALUATION against the frozen thresholds (each model is scored on its own sampled pairs)")
    evaluate(zoo_done, partial=True)


## 4. Full run for ONE model (resumable; copy /kaggle/working/preds_*.jsonl out before the session ends)

In [ ]:
full = WORK / f"preds_{LABEL}_{TAG}.jsonl"
run_model(MODEL, LABEL, full, EXTRA)

In [ ]:
evaluate([full])